In [1]:
import numpy as np

from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('../config/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

print(f"Loaded df of size {df.shape}")

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5
Loaded df of size (33537, 4)


In [2]:
from scripts.utils import extract_unique_npcis
from scripts.data_filter import filter_dataframe

selected_campaigns = list(range(1, 10))

filtered_df = df.copy()

filtered_df = filter_dataframe(
    df=filtered_df,
    operators=[10],
    include_columns=["pci", "beam_index", "nr_arfcn", "operator_id", "rsrp"],
    campaigns=selected_campaigns,
)

print(f"Filtered df of size {filtered_df.shape}")

unique_npcis = extract_unique_npcis(filtered_df['measurements_matrix'])

print(len(unique_npcis))

Filtered df of size (3949, 4)
230


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [4]:
from scripts.utils import RF_PARAM_5G
import pandas as pd
from scripts.beamforming import get_best_beam

# Assuming filtered_df and RF_PARAM_5G are defined elsewhere
data = []
for _, row in filtered_df.iterrows():
    pci, _, _, beam = get_best_beam(row['measurements_matrix'], RF_PARAM_5G.RSRP)
    if pci is not None and beam is not None:
        data.append([row['lat'], row['lng'], pci, beam])

# Convert data into a DataFrame
data_df = pd.DataFrame(data, columns=['lat', 'lng', 'pci', 'beam_index'])


In [87]:
import folium
import matplotlib.pyplot as plt


# Use a discrete colormap with enough distinct colors
def plot_beams(df: pd.DataFrame, title: str) -> None:
    plt.figure(figsize=(12, 8))
    unique_beams = np.unique(df['beam_index'])

    cmap = plt.get_cmap("tab20", 10)  # 'tab20' provides 20 distinct colors

    # Map each unique value to a color
    color_map = {val: cmap(i) for i, val in enumerate(unique_beams)}

    for beam, group in df.groupby('beam_index'):
        plt.scatter(
            group["lat"],
            group["lng"],
            color=color_map[beam],
            label=f"Beam {beam}",
            alpha=0.8,
        )

    plt.xlabel("Latitude")
    plt.ylabel("Longitude")
    plt.title(title)
    plt.legend(title="Beam", loc="lower left")
    plt.show()


def make_html_legend(legend_entries):
    # Create the legend HTML with wider width to accommodate two columns
    legend_html = '''
    <div style="position: fixed;
                bottom: 50px;
                right: 50px;
                width: 300px;
                height: auto;
                background-color: white;
                border: 2px solid grey;
                z-index: 1000;
                padding: 10px;
                font-size: 14px;
                border-radius: 5px;">
        <div style="display: grid;
                    grid-template-columns: 1fr 1fr;
                    gap: 10px;">
    '''

    # Add each legend entry
    for color, symbol, label in legend_entries:
        legend_html += f'''
        <div style="display: flex; align-items: center;">
            <span style="color: {color}; font-weight: bold; font-size: 20px; margin-right: 5px;">{symbol}</span>
            <span>{label}</span>
        </div>
        '''

    legend_html += '</div></div>'

    return legend_html


def geo_plot_points(df: pd.DataFrame, df_sec: pd.DataFrame, title: str) -> None:
    """
    Plots given locations to a map (OpenStreetMap) that is viewable in broswer.
    Generates a file called 'map.html' in the current working directory.
    :param df:
    """
    # Create a map centered around the mean location

    m = folium.Map(location=[df_sec["lat"].mean(), df_sec["lng"].mean()], zoom_start=17)

    unique_beams = list(range(0, 10))
    colors = ['red', 'green', 'blue', 'orange', 'cyan', 'purple', 'teal', 'brown', 'magenta', 'darkgreen']
    color_map = dict(zip(unique_beams, colors))

    # Get all unique beam indices for both datasets
    unique_beams_108 = sorted(df['beam_index'].unique())
    unique_beams_109 = sorted(df_sec['beam_index'].unique())

    # Create dictionaries for legend entries
    legend_entries_108 = {}
    legend_entries_109 = {}

    # Plot PCI -108 beams and store legend entries
    for beam, group in df.groupby('beam_index'):
        color = color_map[beam]
        legend_entries_108[beam] = (color, '▲', f"PCI -108, B-{beam:.0f}")
        print(beam)
        for _, row in group.iterrows():
            folium.RegularPolygonMarker(
                location=[row["lat"], row["lng"]],
                radius=12,
                color=None,
                fill=True,
                fill_color=color,
                fill_opacity=1,
                number_of_sides=3
            ).add_to(m)

    # Plot PCI -109 beams and store legend entries
    for beam, group in df_sec.groupby('beam_index'):
        color = color_map[beam]
        legend_entries_109[beam] = (color, '●', f"PCI -109, B-{beam:.0f}")
        for _, row in group.iterrows():
            folium.CircleMarker(
                location=[row["lat"], row["lng"]],
                radius=8,
                color=None,
                fill=True,
                fill_color=color,
                fill_opacity=0.8,
            ).add_to(m)

    # Combine all unique beam indices
    all_beams = sorted(set(unique_beams_108) | set(unique_beams_109))
    print('all beams:', all_beams)
    # Create alternating legend entries
    legend_entries = []
    for beam in all_beams:
        if beam in legend_entries_108:
            legend_entries.append(legend_entries_108[beam])
        if beam in legend_entries_109:
            legend_entries.append(legend_entries_109[beam])

    for e in legend_entries:
        print(e[2])

    # plot the base stations
    stations = [
        {
            "lat": 41.89666667,
            "lng": 12.42888889,
            "label": "PCI -108"
        },
        {
            "lat": 41.89750000,
            "lng": 12.42638889,
            "label": "PCI -109"
        },
    ]

    folium.RegularPolygonMarker(
        location=[stations[0]['lat'], stations[0]['lng']],
        fill=True,
        color='black',
        fill_color='yellow',
        fill_opacity=1,
        radius=12,
        number_of_sides=3,
    )

    folium.Marker(
        location=[stations[0]['lat'], stations[0]['lng']],
        icon=folium.DivIcon(
            icon_size=(150, 36),
            icon_anchor=(-16, 12),
            html=f'<div style="font-size: 12pt; font-weight: bold;">{stations[0]["label"]}</div>'
        )
    )

    folium.CircleMarker(
        location=[stations[1]['lat'], stations[1]['lng']],
        fill=True,
        color='black',
        fill_color='yellow',
        fill_opacity=1,
        radius=12,
    ).add_to(m)

    folium.Marker(
        location=[stations[1]['lat'], stations[1]['lng']],
        icon=folium.DivIcon(
            icon_size=(150, 36),
            icon_anchor=(-16, 12),
            html=f'<div style="font-size: 12pt; font-weight: bold;">{stations[1]["label"]}</div>'
        )
    ).add_to(m)

    m.get_root().html.add_child(folium.Element(make_html_legend(legend_entries)))
    # Save the map as an HTML file and open it in the browser
    m.save(f"{title}.html")


dfs = []

df_108 = data_df[data_df['pci'] == -11000]
df_109 = data_df[data_df['pci'] == -109]

geo_plot_points(df_108, df_109, title="beams")

unique_beam_indecies = set()

for _, row in df_108.iterrows():
    beams = int(row['beam_index'])
    unique_beam_indecies.add(beams)

print(unique_beam_indecies)

unique_beam_indecies = set()

for _, row in df_109.iterrows():
    beams = int(row['beam_index'])
    unique_beam_indecies.add(beams)

print(unique_beam_indecies)

all beams: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
PCI -109, B-0
PCI -109, B-1
PCI -109, B-2
PCI -109, B-3
PCI -109, B-4
PCI -109, B-5
set()
{0, 1, 2, 3, 4, 5}
